In [1]:
import os
import json
import ast
import operator
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Securely load API key
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
assert api_key, "GOOGLE_API_KEY not set — check your .env file."

# Initialize the Gemini client
client = genai.Client(api_key=api_key)
print("Gemini client ready.")

# 2. Load the orders database
try:
    with open("orders.json", "r") as f:
        orders_db = json.load(f)
    print(f"Successfully loaded {len(orders_db)} orders.")
except FileNotFoundError:
    print("Error: orders.json not found in the current directory.")

Gemini client ready.
Successfully loaded 4 orders.


In [2]:
def lookup_order(order_id: str) -> dict:
    """Looks up an order by its ID and returns its details."""
    # VALIDATION: Ensure the model passed a valid string
    if not isinstance(order_id, str) or not order_id.strip():
        return {"error": "Invalid order_id format. Must be a non-empty string."}
    
    order = orders_db.get(order_id.strip().upper())
    if order:
        return order
    else:
        # Graceful failure for the stretch goal
        return {"error": f"Order '{order_id}' not found in the database."}

def calculate(expression: str) -> dict:
    """Evaluates a basic arithmetic expression safely."""
    # VALIDATION: Ensure the model passed a valid string
    if not isinstance(expression, str) or not expression.strip():
        return {"error": "Invalid expression format. Must be a string."}
    
    # SAFE EVALUATION: Maps ast nodes to safe operator functions
    allowed_operators = {
        ast.Add: operator.add, 
        ast.Sub: operator.sub, 
        ast.Mult: operator.mul, 
        ast.Div: operator.truediv
    }

    def safe_eval_node(node):
        if isinstance(node, ast.Constant): # Handles numbers securely
            if isinstance(node.value, (int, float)):
                return node.value
            raise TypeError("Only numbers are allowed.")
        elif isinstance(node, ast.BinOp):
            return allowed_operators[type(node.op)](safe_eval_node(node.left), safe_eval_node(node.right))
        elif isinstance(node, ast.UnaryOp):
            if isinstance(node.op, ast.USub):
                return -safe_eval_node(node.operand)
            elif isinstance(node.op, ast.UAdd):
                return safe_eval_node(node.operand)
        raise TypeError(f"Unsupported operation or node type: {type(node)}")

    try:
        # Parse the string into an abstract syntax tree and evaluate it
        tree = ast.parse(expression.strip(), mode='eval').body
        result = safe_eval_node(tree)
        return {"result": result}
    except Exception as e:
        return {"error": f"Failed to calculate expression '{expression}': {str(e)}"}

In [3]:
lookup_order_schema = types.FunctionDeclaration(
    name="lookup_order",
    description="Looks up an order by its ID and returns its details, including item name, price, purchase date, and warranty length in months.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "order_id": types.Schema(
                type=types.Type.STRING, 
                description="The exact order ID to look up, e.g., 'A1001'."
            ),
        },
        required=["order_id"]
    )
)

calculate_schema = types.FunctionDeclaration(
    name="calculate",
    description="Evaluates a basic mathematical expression. Use this for all arithmetic to ensure accurate multiplication, addition, subtraction, or division.",
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "expression": types.Schema(
                type=types.Type.STRING, 
                description="The arithmetic expression to evaluate, e.g., '1200 * 3'."
            ),
        },
        required=["expression"]
    )
)

# Package them up for the API
tools = [types.Tool(function_declarations=[lookup_order_schema, calculate_schema])]

In [6]:
from google.genai.errors import ClientError

def run_agent(prompt: str):
    print(f"\nUser: {prompt}\n" + "="*60)
    
    # Start a chat session
    chat = client.chats.create(
        model="gemini-2.5-flash", 
        config=types.GenerateContentConfig(
            tools=tools,
            temperature=0.0 
        )
    )
    
    max_loops = 5
    loop_count = 0
    
    try:
        # Initial request
        response = chat.send_message(prompt)
        
        # THE TOOL-USE LOOP
        while loop_count < max_loops:
            loop_count += 1
            
            # Condition 1: The model wants to call a tool
            if response.function_calls:
                function_responses = []
                
                for tool_call in response.function_calls:
                    tool_name = tool_call.name
                    args = tool_call.args if tool_call.args else {} 
                    
                    print(f"🔄 [Model Request]: call {tool_name}({args})")
                    
                    # Execution Routing
                    if tool_name == "lookup_order":
                        result = lookup_order(args.get("order_id", ""))
                    elif tool_name == "calculate":
                        result = calculate(args.get("expression", ""))
                    else:
                        result = {"error": f"Unknown tool: {tool_name}"}
                        
                    print(f"✅ [System Execution]: returned {result}")
                    
                    function_responses.append(
                        types.Part.from_function_response(
                            name=tool_name,
                            response=result
                        )
                    )
                
                # Send the execution results back to the model
                response = chat.send_message(function_responses)
                
            # Condition 2: The model provided a final text answer
            elif response.text:
                print(f"\n🤖 Final Answer:\n{response.text}")
                break
                
            else:
                print("\n❌ Error: Unexpected response format.")
                break
                
        if loop_count == max_loops:
            print("\n⚠️ Loop aborted: Reached maximum iterations (5).")
            
    except ClientError as e:
        if "429" in str(e):
            print("\n❌ API Rate Limit Reached: You have exhausted your free 20 requests/day limit for gemini-2.5-flash.")
            print("You must wait for the daily quota to reset (usually midnight PT) to run the rest of the lab.")
        else:
            print(f"\n❌ API Error: {str(e)}")

In [7]:
# Test 1: The Two-Tool Question (Requires chaining)
run_agent("For order A1001, what would the total be if I bought three of them?")

# Test 2: The No-Tool Question (Requires directly answering)
run_agent("What can you help me with?")

# Test 3: The Optional Stretch (Requires graceful failure handling)
run_agent("What about order A9999?")


User: For order A1001, what would the total be if I bought three of them?
🔄 [Model Request]: call lookup_order({'order_id': 'A1001'})
✅ [System Execution]: returned {'item': 'laptop', 'price': 1200, 'purchased': '2026-05-20', 'warranty_months': 12}
🔄 [Model Request]: call calculate({'expression': '1200 * 3'})
✅ [System Execution]: returned {'result': 3600}

🤖 Final Answer:
The total for three of order A1001 would be 3600.

User: What can you help me with?

🤖 Final Answer:
I can help you with two things:

1. **Look up an order:** If you provide me with an order ID, I can retrieve its details, including the item name, price, purchase date, and warranty length.
2. **Calculate a mathematical expression:** I can evaluate basic arithmetic expressions for you.

User: What about order A9999?
🔄 [Model Request]: call lookup_order({'order_id': 'A9999'})
✅ [System Execution]: returned {'error': "Order 'A9999' not found in the database."}

🤖 Final Answer:
I'm sorry, but I couldn't find any informa